In [2]:
# ==========================================================
# SECTION 1 : IMPORT REQUIRED LIBRARIES
# ==========================================================

"""
Purpose:
---------
Import all libraries required for data processing,
machine learning model building, evaluation,
and model persistence.
"""

%pip install pandas

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.neighbors import KNeighborsClassifier

from sklearn.naive_bayes import GaussianNB

from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

print("Libraries Imported Successfully")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Libraries Imported Successfully


In [ ]:
# ==========================================================
# SECTION 2 : LOAD DATASET
# ==========================================================

"""
Purpose:
---------
Load the IBM Telco Customer Churn dataset into a
Pandas DataFrame for analysis and model building.
"""

df = pd.read_csv(
    "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

print("Dataset Loaded Successfully")
print("Dataset Shape:", df.shape)

Dataset Loaded Successfully
Dataset Shape: (7043, 21)


In [ ]:
# ==========================================================
# SECTION 3 : DATA AUDIT
# ==========================================================

"""
Purpose:
---------
Perform basic data quality assessment.

Checks Performed:
-----------------
1. Dataset dimensions
2. Duplicate records
3. Target class distribution
4. Hidden missing values in TotalCharges
"""

print("="*60)
print("DATA AUDIT")
print("="*60)

print("\nDataset Shape:")
print(df.shape)

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nTarget Distribution:")
print(df["Churn"].value_counts())

print("\nBlank TotalCharges Values:")
print((df["TotalCharges"] == " ").sum())

DATA AUDIT

Dataset Shape:
(7043, 21)

Duplicate Records:
0

Target Distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Blank TotalCharges Values:
11


In [ ]:
# ==========================================================
# SECTION 4 : DATA PREPROCESSING
# ==========================================================

"""
Purpose:
---------
Prepare the dataset for machine learning.

Steps:
-------
1. Convert TotalCharges to numeric
2. Handle missing values
3. Remove customerID
4. Encode target variable
"""

# Convert TotalCharges
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Fill missing values using median
df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

# Drop Customer ID
df = df.drop(
    "customerID",
    axis=1
)

# Encode target variable
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("Preprocessing Completed Successfully")
print("Final Dataset Shape:", df.shape)

print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

Preprocessing Completed Successfully
Final Dataset Shape: (7043, 20)

Remaining Missing Values:
0


In [ ]:
# ==========================================================
# SECTION 5 : TRAIN-TEST SPLIT
# ==========================================================

"""
Purpose:
---------
Split dataset into training and testing sets.

Why?
----
Training data is used for learning.
Testing data is used for evaluating model performance.

A stratified split is used to maintain
class distribution in both datasets.
"""

X = df.drop(
    "Churn",
    axis=1
)

y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=48,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)

print("\nTraining Target Distribution")
print(y_train.value_counts())

print("\nTesting Target Distribution")
print(y_test.value_counts())

Training Shape: (5282, 19)
Testing Shape : (1761, 19)

Training Target Distribution
Churn
0    3880
1    1402
Name: count, dtype: int64

Testing Target Distribution
Churn
0    1294
1     467
Name: count, dtype: int64


In [ ]:
# ==========================================================
# SECTION 6 : FEATURE IDENTIFICATION
# ==========================================================

"""
Purpose:
---------
Identify numerical and categorical features.

Why?
-----
Different preprocessing techniques are
required for different feature types.
"""

categorical_features = X.select_dtypes(
    include=["object", "str"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object", "str"]
).columns.tolist()

print("="*60)
print("FEATURE IDENTIFICATION")
print("="*60)

print("\nCategorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Count:",
      len(categorical_features))

print("Numerical Count:",
      len(numerical_features))

FEATURE IDENTIFICATION

Categorical Features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Numerical Features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical Count: 15
Numerical Count: 4


In [ ]:
# ==========================================================
# SECTION 7 : PREPROCESSOR CREATION
# ==========================================================

"""
Purpose:
---------
Create preprocessing pipelines for numerical
and categorical features.

Numerical Features:
-------------------
1. Median Imputation
2. Standard Scaling

Categorical Features:
---------------------
1. Most Frequent Imputation
2. One-Hot Encoding

ColumnTransformer combines both pipelines
into one reusable preprocessing workflow.
"""

# Numerical Pipeline
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical Pipeline
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combined Preprocessor

preprocessor = ColumnTransformer([
    ("num",
     numeric_transformer,
     numerical_features),

    ("cat",
     categorical_transformer,
     categorical_features)
])

print("Preprocessor Created Successfully")

Preprocessor Created Successfully


In [ ]:
# ==========================================================
# SECTION 8 : MODEL RESULTS STORAGE
# ==========================================================

"""
Purpose:
---------
Store model evaluation metrics for
all machine learning models.
"""

results = []

print("Results List Created Successfully")

Results List Created Successfully


In [ ]:
# ==========================================================
# MODEL 1 : LOGISTIC REGRESSION
# ==========================================================

"""
Purpose:
---------
Train and evaluate Logistic Regression model.

Why Logistic Regression?
------------------------
1. Baseline classification model.
2. Produces probability scores.
3. Easy to interpret.
4. Suitable for binary classification problems.
"""

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train Model
lr_pipeline.fit(X_train, y_train)

# Predictions
y_pred = lr_pipeline.predict(X_test)

# Probabilities
y_prob = lr_pipeline.predict_proba(X_test)[:, 1]

# Evaluation Metrics
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print("=" * 50)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)

print(f"Accuracy  : {accuracy:.4f}")
print(f"AUC       : {auc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"MCC       : {mcc:.4f}")

results.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy,
    "AUC": auc,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "MCC": mcc
})

LOGISTIC REGRESSION RESULTS
Accuracy  : 0.7973
AUC       : 0.8364
Precision : 0.6403
Recall    : 0.5375
F1 Score  : 0.5844
MCC       : 0.4547


In [ ]:
# ==========================================================
# MODEL 2 : DECISION TREE
# ==========================================================

"""
Purpose:
---------
Train and evaluate Decision Tree Classifier.

Why Decision Tree?
------------------
1. Captures non-linear relationships.
2. Easy to understand and visualize.
3. Handles mixed data effectively.
"""

dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=6,
        random_state=42
    ))
])

dt_pipeline.fit(X_train, y_train)

y_pred = dt_pipeline.predict(X_test)
y_prob = dt_pipeline.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print("="*50)
print("DECISION TREE RESULTS")
print("="*50)

print(f"Accuracy  : {accuracy:.4f}")
print(f"AUC       : {auc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"MCC       : {mcc:.4f}")

results.append({
    "Model":"Decision Tree",
    "Accuracy":accuracy,
    "AUC":auc,
    "Precision":precision,
    "Recall":recall,
    "F1":f1,
    "MCC":mcc
})

DECISION TREE RESULTS
Accuracy  : 0.7825
AUC       : 0.8110
Precision : 0.6055
Recall    : 0.5161
F1 Score  : 0.5572
MCC       : 0.4166


In [ ]:
# ==========================================================
# MODEL 3 : K-NEAREST NEIGHBOR
# ==========================================================

"""
Purpose:
---------
Train and evaluate the KNN classifier.

Why KNN?
--------
1. Distance-based classification algorithm.
2. Works well after feature scaling.
3. Useful benchmark model.
"""

knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(
        n_neighbors=5
    ))
])

knn_pipeline.fit(X_train, y_train)

y_pred = knn_pipeline.predict(X_test)
y_prob = knn_pipeline.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print("="*50)
print("KNN RESULTS")
print("="*50)

print(f"Accuracy  : {accuracy:.4f}")
print(f"AUC       : {auc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"MCC       : {mcc:.4f}")

results.append({
    "Model":"KNN",
    "Accuracy":accuracy,
    "AUC":auc,
    "Precision":precision,
    "Recall":recall,
    "F1":f1,
    "MCC":mcc
})

KNN RESULTS
Accuracy  : 0.7632
AUC       : 0.7752
Precision : 0.5573
Recall    : 0.5203
F1 Score  : 0.5382
MCC       : 0.3796


In [ ]:
# ==========================================================
# MODEL 4 : GAUSSIAN NAIVE BAYES
# ==========================================================

"""
Purpose:
---------
Train and evaluate Gaussian Naive Bayes classifier.

Why Naive Bayes?
----------------
1. Fast probabilistic classifier.
2. Good baseline model.
3. Suitable for classification tasks.
"""

nb_categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

nb_preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", nb_categorical_transformer, categorical_features)
])

nb_pipeline = Pipeline([
    ("preprocessor", nb_preprocessor),
    ("classifier", GaussianNB())
])

nb_pipeline.fit(X_train, y_train)

y_pred = nb_pipeline.predict(X_test)
y_prob = nb_pipeline.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print("="*50)
print("GAUSSIAN NAIVE BAYES RESULTS")
print("="*50)

print(f"Accuracy  : {accuracy:.4f}")
print(f"AUC       : {auc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"MCC       : {mcc:.4f}")

results.append({
    "Model":"Naive Bayes",
    "Accuracy":accuracy,
    "AUC":auc,
    "Precision":precision,
    "Recall":recall,
    "F1":f1,
    "MCC":mcc
})

GAUSSIAN NAIVE BAYES RESULTS
Accuracy  : 0.6917
AUC       : 0.8117
Precision : 0.4561
Recall    : 0.8458
F1 Score  : 0.5926
MCC       : 0.4255


In [ ]:
# ==========================================================
# MODEL 5 : SUPPORT VECTOR MACHINE
# ==========================================================

"""
Purpose:
---------
Train and evaluate Support Vector Machine classifier.

Why SVM?
--------
1. Powerful binary classification algorithm.
2. Effective for high-dimensional datasets.
3. Often achieves strong classification performance.
"""

svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", CalibratedClassifierCV(
        estimator=SVC(random_state=42),
        ensemble=False
    ))
])

# Train Model
svm_pipeline.fit(X_train, y_train)

# Predictions
y_pred = svm_pipeline.predict(X_test)

# Prediction Probabilities
y_prob = svm_pipeline.predict_proba(X_test)[:,1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print("="*50)
print("SUPPORT VECTOR MACHINE RESULTS")
print("="*50)

print(f"Accuracy  : {accuracy:.4f}")
print(f"AUC       : {auc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"MCC       : {mcc:.4f}")

results = [result for result in results if result["Model"] != "SVM"]

results.append({
    "Model":"SVM",
    "Accuracy":accuracy,
    "AUC":auc,
    "Precision":precision,
    "Recall":recall,
    "F1":f1,
    "MCC":mcc
})

SUPPORT VECTOR MACHINE RESULTS
Accuracy  : 0.7899
AUC       : 0.7974
Precision : 0.6622
Recall    : 0.4240
F1 Score  : 0.5170
MCC       : 0.4067


In [ ]:
# ==========================================================
# SECTION 14 : FINAL COMPARISON TABLE
# ==========================================================

"""
Purpose:
---------
Compare all machine learning models using
the mandatory evaluation metrics.
"""

results_df = pd.DataFrame(results)

results_df = results_df.round(4)

print(results_df)

# Save comparison table

results_df.to_csv(
    "model_comparison_results.csv",
    index=False
)

                 Model  Accuracy     AUC  Precision  Recall      F1     MCC
0  Logistic Regression    0.7973  0.8364     0.6403  0.5375  0.5844  0.4547
1        Decision Tree    0.7825  0.8110     0.6055  0.5161  0.5572  0.4166
2                  KNN    0.7632  0.7752     0.5573  0.5203  0.5382  0.3796
3          Naive Bayes    0.6917  0.8117     0.4561  0.8458  0.5926  0.4255
4                  SVM    0.7899  0.7974     0.6622  0.4240  0.5170  0.4067


In [ ]:
# ==========================================================
# SECTION 15 : SAVE TRAINED MODELS
# ==========================================================

import os
import joblib

# The GitHub repo needs a model/ folder holding all five models
os.makedirs("model", exist_ok=True)

saved_models = {
    "logistic_regression_model.joblib": lr_pipeline,
    "decision_tree_model.joblib":       dt_pipeline,
    "knn_model.joblib":                 knn_pipeline,
    "naive_bayes_model.joblib":         nb_pipeline,
    "svm_model.joblib":                 svm_pipeline,
}

for file_name, fitted_pipeline in saved_models.items():
    joblib.dump(fitted_pipeline, os.path.join("model", file_name))

print("Saved:", sorted(os.listdir("model")))

Saved: ['decision_tree_model.joblib', 'knn_model.joblib', 'logistic_regression_model.joblib', 'naive_bayes_model.joblib', 'svm_model.joblib']


In [ ]:
# ==========================================================
# SECTION 15b : CREATE requirements.txt
# ==========================================================

import sklearn, pandas, numpy, joblib, matplotlib

requirements = f"""streamlit
scikit-learn=={sklearn.__version__}
pandas=={pandas.__version__}
numpy=={numpy.__version__}
joblib=={joblib.__version__}
matplotlib=={matplotlib.__version__}
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print(requirements)

streamlit
scikit-learn==1.9.1
pandas==3.1.0
numpy==2.5.4
joblib==1.6.0
matplotlib==3.11.2



In [ ]:
# ==========================================================
# SECTION 16 : EXPORT TEST DATA
# ==========================================================

# Raw feature columns + the Churn column (0 = No, 1 = Yes)
test_data = X_test.copy()
test_data["Churn"] = y_test.values

test_data.to_csv("test_data.csv", index=False)

print("test_data.csv saved:", test_data.shape)
print(test_data["Churn"].value_counts())

test_data.csv saved: (1761, 20)
Churn
0    1294
1     467
Name: count, dtype: int64
